# Pychometrics Tutorial

**Pychometrics** is a Python package for identifying instances of psychological constructs in qualitative text data — such as clinical interviews, therapy transcripts, or open-ended survey responses — using large language models (LLMs).

The core workflow has four steps:

1. **LLM Coding** — run a language model over a directory of interview files to identify and extract quotes that are instances of psychological constructs defined in a codebook
2. **Human Coding** — produce a second set of codings for comparison. In production this is real human-coded data; for testing purposes a second LLM run can serve as a stand-in
3. **Comparison** — align the two sets of codings instance-by-instance using an LLM matcher, classifying each instance as a true positive, false positive, or false negative
4. **Summary Tables** — compute per-interview, concatenated, and weighted performance metrics across constructs

**Prerequisites:**
- An [OpenRouter](https://openrouter.ai) account and API key. OpenRouter provides access to a wide range of models (GPT-4o, Gemini, Claude, Llama, etc.) through a single API. Create an account, add credits, and generate a key at https://openrouter.ai/settings/keys
- A directory of interview files (`.txt` or `.csv`), one file per interview
- A JSON codebook defining the psychological constructs to identify (see `suicide_risk_codebook.json` for an example)

## Setup

Import the core classes and functions used throughout this tutorial:
- `PychometricsAnalyzer` — runs LLM coding over a directory of interview files
- `PychometricsComparator` — aligns and compares two sets of codings (human vs LLM)
- `format_comparison_table` — converts raw comparison output into a row-level DataFrame
- `compute_summary_tables` — computes per-interview, concatenated, and weighted summary metrics
- `format_weighted_summary` — formats the weighted summary table for display
- `format_concatenated` — formats the concatenated table for display

In [ ]:
import pandas as pd

from pychometrics import PychometricsAnalyzer
from pychometrics.comparison import (
    PychometricsComparator,
    format_comparison_table,
    compute_summary_tables,
    format_weighted_summary,
    format_concatenated,
    format_per_interview
)

/Users/freymon.perez/Projects/GitHub/pychometrics/src/pychometrics/models.py:41: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


## Paths and Configuration

Set all paths and credentials here before running any other cells. This is the only cell you need to edit for a new project.

- `input_dir` — directory containing your interview files. Each file should be a single interview in `.txt` or `.csv` format
- `codebook_path` — path to your JSON codebook. This defines the psychological constructs the LLM will look for and extract quotes for
- `api_key` — your OpenRouter API key. **Important:** OpenRouter API keys are case-sensitive — copy the key exactly as shown in your OpenRouter dashboard
- `llm_model` — the model to use for both coding and comparison. Any model listed at https://openrouter.ai/models can be passed here. You can use different models for coding vs comparison if desired (see Step 3)

In [1]:

input_dir     = "/Users/freymon.perez/Downloads/sample_interviews"
codebook_path = "suicide_risk_codebook.json"


# API Key and Model selection.
# Uses OpenRouter. Any model available on OpenRouter can be passed to model_name.
# At the descretion of the user different models can be used for encoding or comparison, 
# In this tutorial we'll stick to one model for all tasks.
api_key   = ""
llm_model = "google/gemini-3-flash-preview"

## Step 1: LLM Coding

This step runs the LLM over every file in `input_dir` and extracts quotes that are instances of each construct defined in the codebook. Each interview is processed independently, and results are saved to a timestamped output directory so runs are never overwritten.

The output directory has the following structure:
```
LLM_coding_YYYY-MM-DD_HHMMSS/
    codings/     <- one JSON per interview, containing extracted construct instances
    metadata/    <- token usage, latency, and model info per document
    errors/      <- any documents that failed, with error messages
    README.md
```

Each coding JSON contains a list of instances, where each instance includes the construct name, the extracted quote, character-level indices into the source text, and a confidence score from the LLM.

**Note:** After running, copy the output directory path from the printout below and update `llm_encoding_dir` in Step 3.

In [6]:
analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

results, metadata, errors = analyzer.analyze_directory(
    input_dir=input_dir,
    codebook_path=codebook_path,
    output_dir="LLM_coding",  # timestamped suffix added automatically
)

Processing [1/3]: s29_integration1.txt
  ✓ Found 13 construct instances
  Progress: Successful 1/3; Errors 0/3
Processing [2/3]: s29_integration2.txt
  ✓ Found 27 construct instances
  Progress: Successful 2/3; Errors 0/3
Processing [3/3]: s7_integration1.txt
  ✓ Found 12 construct instances
  Progress: Successful 3/3; Errors 0/3

Analysis complete!
  Output directory: /Users/freymon.perez/Projects/GitHub/pychometrics/LLM_coding_2026-04-07_031903
  Successful: 3/3


### Retry Failed Documents

If any documents failed — due to API errors, rate limits, or malformed responses — you can retry only the failed ones without re-processing documents that already succeeded. Uncomment the code below and update `output_dir` to point to the timestamped directory from your run above.

In [ ]:

# recovered_results, recovered_meta, remaining_errors = analyzer.retry_errors(
#     output_dir="path/to/my_analysis_1_YYYY-MM-DD_HHMMSS",
#     codebook_path=codebook_path,
# )

## Step 2: Human Coding

For the comparison step you need a second set of codings to compare against the LLM output. In a real study this would be a directory of human-coded JSONs — one per interview, with the same filenames as your input files.

The human coding JSONs must follow the same schema as the LLM coding output: a `document_id` field and an `instances` list where each instance has at minimum a `construct` and `quote` field.

**If you have real human codings:** skip this cell and set `human_encoding_dir` in Step 3 to point directly to your human coding directory.

**If you are testing the pipeline:** run a second LLM coding pass as a stand-in for human data, as shown below. Because the same model is run twice on the same interviews, agreement will be high but not perfect — the LLM introduces some variability between runs, which makes this a reasonable functional test of the full pipeline.

In [7]:
# Example: use a second LLM run as a dummy human coder
human_analyzer = PychometricsAnalyzer(
    api_key=api_key,
    model_name=llm_model,
)

results_human, metadata_human, errors_human = human_analyzer.analyze_directory(
    input_dir=input_dir,
    codebook_path=codebook_path,
    output_dir="human_coding",
)

Processing [1/3]: s29_integration1.txt
  ✓ Found 14 construct instances
  Progress: Successful 1/3; Errors 0/3
Processing [2/3]: s29_integration2.txt
  ✓ Found 25 construct instances
  Progress: Successful 2/3; Errors 0/3
Processing [3/3]: s7_integration1.txt
  ✓ Found 10 construct instances
  Progress: Successful 3/3; Errors 0/3

Analysis complete!
  Output directory: /Users/freymon.perez/Projects/GitHub/pychometrics/human_coding_2026-04-07_032120
  Successful: 3/3


## Step 3: Comparison

This step aligns the human and LLM codings for each interview. For each construct, an LLM matcher reads all quotes from both coders and decides which human quotes and LLM quotes refer to the same passage or idea — even if the wording differs (paraphrase).

Each instance in the resulting DataFrame is classified as one of three statuses:
- **`matched`** — both coders identified this passage (True Positive, TP). Includes a `match_confidence` score from the matcher and a `span_overlap` score (Jaccard overlap of character indices)
- **`human_only`** — the human identified this passage but the LLM missed it (False Negative, FN)
- **`llm_only`** — the LLM identified this passage but the human missed it (False Positive, FP)

**Important:** update `llm_encoding_dir` and `human_encoding_dir` below to the `codings/` subdirectories of your runs from Steps 1 and 2. The paths include timestamps so they will be different each time you run.

You can optionally use a different model for the matching step than for coding — for example a more capable model for matching if accuracy is critical. Set `match_model` accordingly.

In [4]:
# Update these paths to point to the codings/ subdirectories of your runs
# If this notebook is run multiple times, remember to update these paths.
llm_encoding_dir   = "/Users/freymon.perez/Projects/GitHub/pychometrics/LLM_coding_2026-04-07_031903/encodings"
human_encoding_dir = "/Users/freymon.perez/Projects/GitHub/pychometrics/human_coding_2026-04-07_032120/encodings"
#change the match model if you want different models for encoding and comparison. 
comparator = PychometricsComparator(
    api_key=api_key,
    match_model=llm_model, 
)

comparison_results = comparator.compare_directories(
    human_dir=human_encoding_dir,
    llm_dir=llm_encoding_dir,
    output_dir="comparison_run",  # optional; saves JSON comparison files
)

## Step 4: Summary Tables

This step builds a combined row-level DataFrame across all interviews and computes three summary tables, each offering a different view of LLM coding performance.

| Table | Grain | Description |
|---|---|---|
| `per_interview` | (interview, construct) | Raw TP/FP/FN counts and all metrics for each interview-construct combination |
| `concatenated` | construct | Counts pooled across all interviews, metrics computed from totals |
| `weighted_summary` | construct | Per-interview metrics summarized as weighted median [min–max] across interviews, weighted by union size |

The following metrics are computed for each construct in each table:

- **Sensitivity** (Recall) — `TP / (TP + FN)`. Of all instances the human identified, what proportion did the LLM also find?
- **Precision** — `TP / (TP + FP)`. Of all instances the LLM identified, what proportion were correct?
- **F1** — harmonic mean of sensitivity and precision. A single balanced measure of overall coding performance
- **Cohen's Kappa** — agreement between coders beyond what would be expected by chance. Computed over the union of coded instances (TP + FP + FN), with true negatives set to zero — this is the appropriate convention for open-ended span coding tasks where there is no fixed inventory of candidate passages. As a result kappa values will tend to be lower than in fixed-item rating tasks and should be interpreted comparatively across constructs rather than against standard benchmarks
- **PABAK** (Prevalence-Adjusted Bias-Adjusted Kappa) — a variant of kappa that is more stable when one class is rare. Computed as `2 * observed_agreement - 1`. Useful alongside kappa when construct prevalence varies substantially across interviews
- **PR AUC** (Precision-Recall Area Under the Curve) — ranks all LLM predictions by `match_confidence` and measures how well that ranking separates correct predictions (matched) from incorrect ones (llm_only). A value of 1.0 means perfect ranking; 0.5 is chance. 

In [5]:
# Combine all interview comparison results into one DataFrame
df = pd.concat(
    [format_comparison_table(result) for result in comparison_results.values()],
    ignore_index=True,
)

per_interview, concatenated, weighted_summary = compute_summary_tables(df)

### 4.1 Row-Level Comparison DataFrame

The raw DataFrame `df` has one row per coded instance across all interviews. Each row represents a single quote that was identified by at least one coder, and includes the full quote text, character indices, and TP/FP/FN indicators. This is the most granular view and is useful for inspecting specific mismatches — for example, filtering to `status == 'llm_only'` to review what the LLM found that the human missed, or `status == 'human_only'` to review misses.

In [6]:
# Row-level view: every matched/human_only/llm_only instance across all interviews
df

,doc_id,construct,status,human_quote,llm_quote,human_indices,llm_indices,paraphrase,span_overlap,match_confidence,tp,fp,fn
0,s29_integration1,anxiety,matched,"I was a bit excited and anxious, I guess.","I was a bit excited and anxious, I guess.",273:314,273:314,False,1.000000,1.0,1,0,0
1,s29_integration1,anxiety,llm_only,NaN,And that it's not immediate. And it takes time...,NaN,497:607,None,NaN,NaN,0,1,0
2,s29_integration1,anxiety,llm_only,NaN,"I was a bit frightened that I'm going to, like...",NaN,1316:1447,None,NaN,NaN,0,1,0
3,s29_integration1,depressed_mood,matched,I maybe felt a bit sad about it or just... but...,I maybe felt a bit sad about it or just... but...,9784:9874,9784:9874,False,1.000000,1.0,1,0,0
4,s29_integration1,depressed_mood,human_only,if I wake up with a sense of tenderness or vul...,NaN,17192:17278,NaN,None,NaN,NaN,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
64,s7_integration1,rumination,human_only,I think it's something that I want to think ab...,NaN,21488:21627,NaN,None,NaN,NaN,0,0,1
65,s7_integration1,rumination,llm_only,NaN,I've picked it apart and it doesn't make sense.,NaN,16534:16581,None,NaN,NaN,0,1,0
66,s7_integration1,shame,llm_only,NaN,"when you're having a dream, and you make a tit...",NaN,14176:14264,None,NaN,NaN,0,1,0
67,s7_integration1,shame,llm_only,NaN,I woke up really embarrassed.,NaN,14455:14484,None,NaN,NaN,0,1,0


### 4.2 Per-Interview Metrics

One row per (interview, construct) combination. Shows raw TP/FP/FN counts and all metrics broken down by document. Use this table to identify whether performance issues are consistent across interviews or driven by specific documents. Constructs that do not appear in a given interview are absent from this table — they do not appear as zero rows. PR AUC is computed per (interview, construct) where there are enough positive and negative predictions; otherwise it is `NaN`.

In [ ]:
# Per-interview metrics: one row per (interview, construct)
format_per_interview(per_interview)

,doc_id,construct,tp,fp,fn,union,sensitivity,precision,f1,cohens_kappa,pabak,pr_auc
0,s29_integration1,anxiety,1,2,0,3,1.00,0.33,0.50,0.00,-0.33,1.0
1,s29_integration1,depressed_mood,1,0,1,2,0.50,1.00,0.67,0.00,0.00,NaN
2,s29_integration1,guilt,0,0,1,1,0.00,NaN,0.00,0.00,-1.00,NaN
3,s29_integration1,panic,0,0,1,1,0.00,NaN,0.00,0.00,-1.00,NaN
4,s29_integration1,rumination,1,0,0,1,1.00,1.00,1.00,1.00,1.00,NaN
5,s29_integration1,shame,4,4,4,12,0.50,0.50,0.50,-0.50,-0.33,1.0
6,s29_integration2,anxiety,5,4,0,9,1.00,0.56,0.71,0.00,0.11,1.0
7,s29_integration2,depressed_mood,5,2,5,12,0.50,0.71,0.59,-0.31,-0.17,1.0
8,s29_integration2,panic,2,1,1,4,0.67,0.67,0.67,-0.33,0.00,1.0
9,s29_integration2,rumination,1,1,0,2,1.00,0.50,0.67,0.00,0.00,1.0


### 4.3 Concatenated Metrics

One row per construct, with counts pooled across all interviews before metrics are computed. Think of this as treating the entire dataset as a single document. This table answers: *overall, across all interviews, how did the LLM perform on each construct?* The `number of docum [p5–p95]` column shows how many interviews contained at least one instance of the construct, along with the 5th and 95th percentile of instance counts — giving a sense of how common and variable the construct is across your data.

In [8]:
# Concatenated metrics: counts pooled across all interviews, one row per construct
format_concatenated(concatenated)

,construct,tp,fp,fn,sensitivity,precision,f1,cohens_kappa,pabak,pr_auc,n_docs [p5-p95]
0,anxiety,7,6,0,1.00,0.54,0.70,0.00,0.08,1.00,3 [1.20-8.40]
1,depressed_mood,6,2,6,0.50,0.75,0.60,-0.27,-0.14,1.00,2 [0.20-11.00]
2,guilt,4,1,2,0.67,0.80,0.73,-0.24,0.14,1.00,2 [0.10-5.50]
3,panic,2,1,2,0.50,0.67,0.57,-0.36,-0.20,1.00,2 [0.10-3.70]
4,rumination,4,2,1,0.80,0.67,0.73,-0.24,0.14,1.00,3 [1.10-3.80]
5,shame,7,8,5,0.58,0.47,0.52,-0.44,-0.30,1.00,3 [2.40-11.40]
6,trauma_and_ptsd,2,0,1,0.67,1.00,0.80,0.00,0.33,—,2 [0.10-1.90]
7,Overall,32,20,17,0.65,0.62,0.63,-0.36,-0.07,1.00,3 [0.00-12.00]


### 4.4 Weighted Summary

One row per construct. Metrics are first computed per interview, then summarized as weighted median [min–max] across interviews, where each interview is weighted by its union size (TP + FP + FN) — so interviews with more coded instances contribute more to the median. This table answers: *what does performance typically look like at the individual interview level?* The [min–max] brackets show the range across interviews, which is useful for spotting constructs with highly variable performance. A `—` in the PR AUC column means there were insufficient label classes in every interview to compute the metric for that construct.

In [9]:
# Weighted summary: median [min-max] weighted by union size, with n_docs prevalence column
format_weighted_summary(weighted_summary)

,construct,tp,fp,fn,sensitivity,precision,f1,cohens_kappa,pabak,pr_auc,n_docs [p5–p95]
0,anxiety,7,6,0,1.00 [1.00–1.00],0.56 [0.33–1.00],0.71 [0.50–1.00],0.00 [0.00–1.00],0.11 [-0.33–1.00],1.00 [1.00–1.00],3 [1.20–8.40]
1,depressed_mood,6,2,6,0.50 [0.50–0.50],0.71 [0.71–1.00],0.59 [0.59–0.67],-0.31 [-0.31–0.00],-0.17 [-0.17–0.00],1.00 [1.00–1.00],2 [0.20–11.00]
2,guilt,4,1,2,0.80 [0.00–0.80],0.80 [0.80–0.80],0.80 [0.00–0.80],-0.20 [-0.20–0.00],0.33 [-1.00–0.33],1.00 [1.00–1.00],2 [0.10–5.50]
3,panic,2,1,2,0.67 [0.00–0.67],0.67 [0.67–0.67],0.67 [0.00–0.67],-0.33 [-0.33–0.00],0.00 [-1.00–0.00],1.00 [1.00–1.00],2 [0.10–3.70]
4,rumination,4,2,1,0.67 [0.67–1.00],0.67 [0.50–1.00],0.67 [0.67–1.00],-0.33 [-0.33–1.00],0.00 [0.00–1.00],1.00 [1.00–1.00],3 [1.10–3.80]
5,shame,7,8,5,0.50 [0.50–0.75],0.50 [0.00–0.60],0.50 [0.00–0.67],-0.50 [-0.50–0.00],-0.33 [-1.00–0.00],1.00 [1.00–1.00],3 [2.40–11.40]
6,trauma_and_ptsd,2,0,1,0.50 [0.50–1.00],1.00 [1.00–1.00],0.67 [0.67–1.00],0.00 [0.00–1.00],0.00 [0.00–1.00],—,2 [0.10–1.90]
7,Overall,32,20,17,0.67 [0.00–1.00],0.60 [0.00–1.00],0.67 [0.00–1.00],-0.29 [-0.50–1.00],0.00 [-1.00–1.00],1.00 [1.00–1.00],3 [0.00–12.00]
